#### Application Codebase Knowledge Graph (KG) Tools

##### Environment setup

###### Package imports

In [20]:
import os
import sys
from dotenv import load_dotenv
import time
import requests
import pandas as pd
from pathlib import Path
import importlib
from neo4j import GraphDatabase
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget
import gradio as gr
import gradio.blocks

###### Test connections

In [2]:
def test_neo4j_connection(env_file=Path.cwd().parent / ".env"):
    """
    Test connectivity to a Neo4j database using settings
    stored in a .env file.

    Expected variables:

        NEO4J_URI
        NEO4J_USERNAME
        NEO4J_PASSWORD
        NEO4J_DATABASE
    """

    # Load environment variables
    load_dotenv(env_file, override=True)

    uri = os.getenv("NEO4J_URI")
    username = os.getenv("NEO4J_USERNAME")
    password = os.getenv("NEO4J_PASSWORD")
    database = os.getenv("NEO4J_DATABASE")

    try:
        with GraphDatabase.driver(uri, auth=(username, password), database=database) as driver:

            # Test the connection
            driver.verify_connectivity()

            # Open a session
            with driver.session(database=database) as session:
                result = session.run(
                    """
                    RETURN
                        1 AS connected,
                        datetime() AS server_time
                    """
                )

                record = result.single()

                print("✅ Successfully connected to Neo4j")
                print(f"Database    : {database}")
                print(f"URI         : {uri}")
                print(f"Server Time : {record['server_time']}")

                return True

    except Exception as ex:
        print("❌ Connection failed")
        print(type(ex).__name__)
        print(ex)

        return False

test_neo4j_connection()

✅ Successfully connected to Neo4j
Database    : appkg
URI         : neo4j://127.0.0.1:7687
Server Time : 2026-09-16T02:05:41.256000000+00:00


True

###### Create Neo4j driver object and python-cypher method

In [10]:
# Load .env variables
#uri = os.getenv("NEO4J_URI")
#username = os.getenv("NEO4J_USERNAME")
#password = os.getenv("NEO4J_PASSWORD")
#database = os.getenv("NEO4J_DATABASE")

uri = "neo4j://127.0.0.1:7687"
username = "neo4j"
password = "SuperNeo4j1"
database = "appkg"

# Create driver object
# Keeping the Neo4j Driver open for the lifetime of your Jupyter session is the normal pattern and does not, by itself, constitute a memory leak.
driver = GraphDatabase.driver(uri, auth=(username, password), database=database)

driver.verify_connectivity()

# Define a generic function to run a Cypher query
def run_cypher_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return result.data()

###### Test Github API connection

In [3]:
def test_github_connection():
    """Test connectivity to GitHub and validate the GitHub PAT."""

    # Find .env in the directory above the notebook's working directory
    env_file = Path.cwd().parent / ".env"

    print(f"Loading environment from: {env_file}")

    if not env_file.exists():
        print("❌ .env file not found")
        return False

    load_dotenv(env_file, override=True)

    token = os.getenv("GITHUB_TOKEN")

    if not token:
        print("❌ GITHUB_TOKEN is missing from .env")
        return False

    # GitHub API headers
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2026-03-10",
    }

    try:
        # Test GitHub API + PAT authentication
        response = requests.get(
            "https://api.github.com/user",
            headers=headers,
            timeout=10,
        )

        print(f"GitHub API status: {response.status_code}")

        if response.status_code == 200:
            user = response.json()

            print("✅ Successfully connected to GitHub")
            print("✅ PAT is valid")
            print(f"GitHub user : {user.get('login')}")
            print(f"User ID     : {user.get('id')}")
            print(f"Account type: {user.get('type')}")

            return True

        elif response.status_code == 401:
            print("❌ GitHub API connection succeeded")
            print("❌ PAT authentication failed")
            print("The token may be invalid, expired, or revoked.")

        elif response.status_code == 403:
            print("⚠️ GitHub API responded with 403 Forbidden")
            print("The PAT was received, but GitHub denied the request.")
            print(response.text)

        else:
            print("❌ GitHub API request failed")
            print(response.text)

        return False

    except requests.exceptions.Timeout:
        print("❌ Connection to GitHub timed out")
        return False

    except requests.exceptions.ConnectionError as ex:
        print("❌ Could not connect to GitHub")
        print(ex)
        return False

    except requests.exceptions.RequestException as ex:
        print("❌ GitHub request failed")
        print(type(ex).__name__)
        print(ex)
        return False

test_github_connection()

Loading environment from: E:\Projects\knowledge_graph_tools\.env
GitHub API status: 200
✅ Successfully connected to GitHub
✅ PAT is valid
GitHub user : georgejaymcmc
User ID     : 88515517
Account type: User


True

##### Repo details

In [4]:
# 1. Enable auto-reloading so changes to your .py file update automatically
%load_ext autoreload
%autoreload 2

# 2. Add your project 'src' directory to Python's path
# Adjust the number of parents depending on where your notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parents[0] # Assumes notebook is one folder deep (e.g., in a 'notebooks/' folder)
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 3. Import your service directly from your local module
from codebase_kg.services.github_repo_service import GithubRepoService

# 4. Initialize and use it
# service = GithubRepoService(token="YOUR_GITHUB_TOKEN")
print("Module successfully imported from:", sys.modules['codebase_kg.services.github_repo_service'].__file__)


Module successfully imported from: E:\Projects\knowledge_graph_tools\src\codebase_kg\services\github_repo_service.py


##### Extracting Zotero repo stats

###### Code: Choose repo to analyse
1. Fork repo to baseline codebase - 16/09/2026

In [5]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "georgejaymcmc"
repo = "zotero"

# Print result
result = github.stats(owner, repo)

# Pretty print result
for key, value in result.items():
    print(f"{key:20}: {value}")


name                : zotero
full_name           : georgejaymcmc/zotero
description         : Zotero is a free, easy-to-use tool to help you collect, organize, annotate, cite, and share your research sources.
language            : None
size_kb             : 242678
visibility          : public
default_branch      : main
stars               : 0
forks               : 0
watchers            : 0
directories         : 363
files               : 3489
url                 : https://github.com/georgejaymcmc/zotero


###### Code: Purge the cached module from the notebook

In [7]:
# 1. Collect all loaded submodules related to your project package
to_delete = [name for name in sys.modules if name.startswith("codebase_kg")]

# 2. Purge them from the sys.modules memory cache
for module_name in to_delete:
    del sys.modules[module_name]
    print(f"🔥 Purged cache for: {module_name}")

# 3. Re-import your updated class freshly from the disk
from codebase_kg.services.github_repo_service import GithubRepoService


🔥 Purged cache for: codebase_kg
🔥 Purged cache for: codebase_kg.services
🔥 Purged cache for: codebase_kg.services.github_repo_service


###### Code: Output repo file types and directory location to csv

In [8]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "georgejaymcmc"
repo = "zotero"

# 1. RUN THIS METHOD TO GENERATE THE CSV FILE
print("Generating CSV file...")
csv_results = github.file_types(owner, repo, output_csv="zotero_files.csv")

# 2. Pretty print the execution summary returned by file_types
print("\n--- Execution Summary ---")
for key, value in csv_results.items():
    print(f"{key:20}: {value}")


Generating CSV file...
📦 Dataframe successfully dumped to: E:\Projects\knowledge_graph_tools\src\codebase_kg\neo4j_imports\zotero_files.csv

--- Execution Summary ---
total_files         : 3489
directory_count     : 271
output_csv          : zotero_files.csv
file_type_counts    : {'SVG': 774, 'FreeMarker_Java_template': 673, 'Javascript': 578, 'DTD_XML_def': 482, 'CSS': 202, 'Java_prop': 146, 'Other': 108, 'Header': 107, 'C++': 77, 'JSON': 63, 'XHTML': 51, 'PNG': 41, 'HTML': 23, 'PDF': 18, 'Text': 16, 'Typescript': 14, 'Shell': 12, 'IDL': 10, 'NSH': 7, 'GIF': 7, 'RSS': 7, 'Epub': 7, 'Markdown': 6, 'RC': 6, 'XUL': 6, 'NSI': 5, 'XML': 4, 'Python Script': 4, 'ICO': 4, 'SQL': 4, '.ini': 3, '.manifest': 3, 'CSL': 3, 'VBScript': 2, 'WOFF': 2, 'SQLite': 2, 'RDF': 2, 'YAML': 1, 'C': 1, 'NLF': 1, 'XPI': 1, 'ATOM': 1, 'OPML': 1, 'LUA': 1, 'OPF': 1, 'JPG': 1, 'ZIP': 1}
unknown_extensions  : ['', '.0_release_build_and_deploy', '.car', '.ctv6', '.def', '.desktop', '.dsp', '.dsw', '.mts', '.patch', 

##### Import what we know into Neo4j: KGs are built progressively
- directories, file names and file types form the base of the App KG

###### Start from scratch: Delete all nodes, relationships, etc from Neo4j

In [16]:
# Delete all nodes and relationships

query_delete_all_nodes_relationships = """
MATCH (n)
DETACH DELETE n
RETURN count(n) AS deleted_nodes
"""

result_query_delete_all_nodes_relationships = run_cypher_query(query_delete_all_nodes_relationships)

result_query_delete_all_nodes_relationships

[{'deleted_nodes': 0}]

###### Upload repo_files.csv into Neo4j to create base graph elements

In [17]:
# Generic function to run Cypher queries
query_load_repo_files = """
// Load File Type, Directory, and File Name nodes
LOAD CSV WITH HEADERS FROM 'file:///zotero_files.csv' AS row
MERGE (f:FileName {name: row.FileName})
MERGE (d:Directory {name: row.Directory})
MERGE (t:FileType {type: row.FileType})
MERGE (d)-[:CONTAINS]->(f)
MERGE (f)-[:IS_TYPE_OF]->(t);
"""
result_query_load_repo_files = run_cypher_query(query_load_repo_files)

###### Check the upload
- Compare file count with above repo query result

In [18]:
# Generate graph from Cypher query
g = Neo4jGraphWidget(driver)
# View schema
g.show_cypher("CALL db.schema.visualization()")

query_node_frequency = """
MATCH (n)
RETURN labels(n) AS label, count(*) AS count
ORDER BY count DESC;
"""
result_query_node_frequency = run_cypher_query(query_node_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_nodes = pd.DataFrame(result_query_node_frequency)
# Flatten the 'label' column since it's a list
df_nodes['label'] = df_nodes['label'].apply(
    lambda x: x[0] if len(x) > 0 else None)
print("Nodes by Count")
print(df_nodes)  # Outputs the list of nodes (Persons)

query_relationship_frequency = """
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, COUNT(r) AS count
ORDER BY count DESC;
"""
result_query_relationship_frequency = run_cypher_query(
    query_relationship_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_rels = pd.DataFrame(result_query_relationship_frequency)
# Flatten the 'label' column since it's a list
# df_rels['label'] = df_rels['label'].apply(lambda x: x[0] if len(x) > 0 else None)
print("Relationships by Count")
print(df_rels)  # Outputs the list of nodes (Persons)


GraphWidget(layout=Layout(height='500px', width='100%'))

Nodes by Count
       label  count
0   FileName   1634
1  Directory    271
2   FileType     47
Relationships by Count
  relationshipType  count
0         CONTAINS   3489
1       IS_TYPE_OF   1634


##### Gradio chatbot file selector

###### Function to list labels and relationship types

In [ ]:
def get_graph_schema():
    """Return all node labels and relationship types currently in the graph."""

    try:
        with driver.session(database=database) as session:

            # Node labels
            labels_result = session.run("""
                MATCH (n)
                UNWIND labels(n) AS label
                RETURN DISTINCT label
                ORDER BY label
            """)

            labels = [record["label"] for record in labels_result]

            # Relationship types
            relationships_result = session.run("""
                MATCH ()-[r]->()
                RETURN DISTINCT type(r) AS relationship
                ORDER BY relationship
            """)

            relationships = [
                record["relationship"]
                for record in relationships_result
            ]

        output = "NODE LABELS\n"
        output += "-----------\n"

        if labels:
            output += "\n".join(f"• {label}" for label in labels)
        else:
            output += "(none)"

        output += "\n\nRELATIONSHIP TYPES\n"
        output += "------------------\n"

        if relationships:
            output += "\n".join(
                f"• {relationship}"
                for relationship in relationships
            )
        else:
            output += "(none)"

        return output

    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

###### Gradio interface

In [21]:
def get_graph_schema():
    """Return node labels and relationship types with counts."""

    try:

        # Node labels
        labels = run_cypher_query("""
            MATCH (n)
            UNWIND labels(n) AS label
            RETURN label, count(*) AS count
            ORDER BY label
        """)

        # Relationship types
        relationships = run_cypher_query("""
            MATCH ()-[r]->()
            RETURN type(r) AS relationship, count(*) AS count
            ORDER BY relationship
        """)

        # Format output
        output = "NODE LABELS\n"
        output += "-----------\n"

        if labels:
            for row in labels:
                output += (
                    f"{row['label']:<30} "
                    f"{row['count']:>10,}\n"
                )
        else:
            output += "(none)\n"

        output += "\nRELATIONSHIP TYPES\n"
        output += "------------------\n"

        if relationships:
            for row in relationships:
                output += (
                    f"{row['relationship']:<30} "
                    f"{row['count']:>10,}\n"
                )
        else:
            output += "(none)\n"

        return output

    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

In [24]:
def execute_cypher(query):
    """Execute Cypher and return the results as a DataFrame."""

    if not query or not query.strip():
        return pd.DataFrame({
            "Result": ["Please enter a Cypher query."]
        })

    try:
        records = run_cypher_query(query)

        if not records:
            return pd.DataFrame({
                "Result": ["Query returned no rows."]
            })

        return pd.DataFrame(records)

    except Exception as e:
        return pd.DataFrame({
            "Error": [
                f"{type(e).__name__}: {e}"
            ]
        })

In [30]:
with gr.Blocks(title="Neo4j Cypher Explorer") as demo:

    gr.Markdown("# Neo4j Cypher Explorer")
    gr.Markdown("Explore the current graph model and execute Cypher queries.")

    # ---------------------------------------------------------
    # Graph schema
    # ---------------------------------------------------------

    schema_button = gr.Button(
        "Refresh Graph Schema",
        variant="secondary"
    )

    schema_output = gr.Textbox(
        label="Graph Schema",
        lines=15,
        interactive=False
    )

    # ---------------------------------------------------------
    # Cypher query
    # ---------------------------------------------------------

    gr.Markdown("## Cypher Query")

    cypher_input = gr.Textbox(
        label="Enter Cypher",
        lines=8,
        value="MATCH (n) RETURN labels(n) AS labels, properties(n) AS properties LIMIT 10"
    )

    cypher_input.submit(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )

    execute_button = gr.Button(
        "Run Query",
        variant="primary"
    )

    # ---------------------------------------------------------
    # Results
    # ---------------------------------------------------------

    gr.Markdown("## Results")

    result_output = gr.Dataframe(
        label="Query Result",
        interactive=False,
        wrap=True
    )

    # ---------------------------------------------------------
    # Events
    # ---------------------------------------------------------

    schema_button.click(
        fn=get_graph_schema,
        outputs=schema_output
    )

    execute_button.click(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [29]:
query_test="""
MATCH (n)
RETURN n LIMIT 10
"""

result_query_test = run_cypher_query(query_test)

result_query_test

[{'n': {'name': '.babelrc'}},
 {'n': {'name': '.gitattributes'}},
 {'n': {'name': 'ci.yml'}},
 {'n': {'name': '.gitignore'}},
 {'n': {'name': '.gitmodules'}},
 {'n': {'name': 'CLAUDE.md'}},
 {'n': {'name': 'CONTRIBUTING.md'}},
 {'n': {'name': 'COPYING'}},
 {'n': {'name': 'README.md'}},
 {'n': {'name': 'application.ini'}}]